In [1]:
import pandas as pd
import numpy as np

Vds_max = 12
Vth = 3
Ta = 30 # Temperatura ambiente en grados Celsius
Rt_jc = 1.1 # grados C / W
Rt_cs = 0.5 # grados C / W
Rt_ja = 62 # grados C / W

Rt_ca = Rt_ja - Rt_jc - Rt_cs
print("Resistencia termica del disipador:", Rt_ca, "C/W")

r_norm = pd.Series([1000, 1200, 1500, 2200, 3300, 4700, 5100, 6800])
r1 = r_norm

Resistencia termica del disipador: 60.4 C/W


## Calculos rangos de resistencias - MOSFET

In [ ]:
Io_max = 250e-3
Vgs_max = 18

Vgs_on = np.array([4,4.5,5,6,6.5])
Id_on = np.array([6,11,17,25,40])
b_list = Id_on / ((Vgs_on - Vth) ** 2)
b = b_list.sum() / b_list.size
print(f"El valor de b es: {b:.2f} A/V^2")

def get_current(vgs, vds):
    if vgs < Vth:
        return 0
    return b * (2 * (vgs - Vth) * vds - vds ** 2)

vgs_values = np.arange(3, Vgs_max + 1, 0.5)

def get_vds_values(vds: float):
    df = pd.DataFrame(
        {
            "Vgs": vgs_values,
            "Id": [get_current(vgs, vds) for vgs in vgs_values],
            "Vds": vds
        }
    )
    df["Rds"] = vds / df["Id"].replace(0, np.nan)
    return df
get_vds_values(5.0)


El valor de b es: 4.24 A/V^2


,Vgs,Id,Vds,Rds
0,3.0,-105.909864,5.0,-0.047210
1,3.5,-84.727891,5.0,-0.059012
2,4.0,-63.545918,5.0,-0.078683
3,4.5,-42.363946,5.0,-0.118025
4,5.0,-21.181973,5.0,-0.236050
5,5.5,0.000000,5.0,NaN
6,6.0,21.181973,5.0,0.236050
7,6.5,42.363946,5.0,0.118025
8,7.0,63.545918,5.0,0.078683
9,7.5,84.727891,5.0,0.059012


## INA219 Registers

In [5]:
Iomax = 0.5 # Amperios
Rshunt = 0.1 # Ohmios
current_lsb = Iomax / 2**15
print(f"Corriente LSB: {current_lsb:.8f}")
calibration_bit = 0.04096 / (current_lsb * Rshunt)
print(f"Calibracion del bit: {calibration_bit:.8f}")

Corriente LSB: 0.00001526
Calibracion del bit: 26843.54560000


## Calculo de disipador del IRF540N
[datasheet](https://www.farnell.com/datasheets/67691.pdf)

In [7]:
df_disipador = pd.DataFrame([250e-3, 350e-3, 500e-3, 750e-3, 1, 1.25], columns=['Imax [A]'])
df_disipador["Pdis [W]"] = df_disipador["Imax [A]"] * Vds_max

df_disipador["Tj"] = Ta + df_disipador["Pdis [W]"] * Rt_ja

Rth_sa_6225 = 8.8 + Rt_cs# grados C / W
Rth_eq_6225 = Rt_ca * Rth_sa_6225 / (Rt_ca + Rth_sa_6225)
print(f"Resistencia termica equivalente del disipador: {Rth_eq_6225:.2f}C/W")

Rth_sa_2725 = 10 + Rt_cs# grados C / W
Rth_eq_2725 = Rt_ca * Rth_sa_2725 / (Rt_ca + Rth_sa_2725)
print(f"Resistencia termica equivalente del disipador: {Rth_eq_2725:.2f}C/W")


df_disipador["Tj_eq_6225"] = Ta + df_disipador["Pdis [W]"] * Rth_eq_6225
df_disipador["Tj_eq_2725"] = Ta + df_disipador["Pdis [W]"] * Rth_eq_2725

df_disipador


Resistencia termica equivalente del disipador: 8.06C/W
Resistencia termica equivalente del disipador: 8.94C/W


,Imax [A],Pdis [W],Tj,Tj_eq_6225,Tj_eq_2725
0,0.25,3.0,216.0,54.177331,56.834979
1,0.35,4.2,290.4,63.848264,67.568970
2,0.50,6.0,402.0,78.354663,83.669958
3,0.75,9.0,588.0,102.531994,110.504937
4,1.00,12.0,774.0,126.709326,137.339915
5,1.25,15.0,960.0,150.886657,164.174894


## Op Amp Calculos

In [52]:
Vgs_max = 16
v_lpf_max = 3.3
gain_amp = Vgs_max / v_lpf_max
print(f"Ganacia objetivo: {gain_amp:.2f}")
df = pd.DataFrame(r_norm, columns=['R1_norm'])
df["R2"] = (gain_amp - 1) * df["R1_norm"]
display(df)

vout_test = v_lpf_max * (1 + 20e3 / 5.1e3)
vout_test

Ganacia objetivo: 4.85


,R1_norm,R2
0,1000,3848.484848
1,1500,5772.727273
2,2200,8466.666667
3,3300,12700.000000
4,4700,18087.878788
5,5100,19627.272727
6,6800,26169.696970


16.241176470588236

## Fuente Boost para Op Amp
Se utiliza el [MC34063](https://www.ti.com/lit/ds/symlink/mc34063a.pdf)


In [19]:
v_in = 5.0      # Voltaje de entrada
v_out = 19.0    # Voltaje de salida
i_out = 0.5     # Corriente de salida
f0 = 25 * 1000  # Frecuencia de operación
t0 = 1 / f0     # Periodo de operación
v_ripple = 0.1  # Voltaje de rizado

$\frac{ton}{toff}=A$ -> $ton=A*(T - ton)$ -> $ton = \frac{A*T}{1+A}$


In [20]:
def mc34063_calculations(v_in, v_out, i_out, f0, v_ripple):
    """
    Calcula los componentes necesarios para el circuito boost con MC34063.
    """
    VF = 0.4
    VSAT = 1.42
    t0 = 1 / f0
    ton_toff = (v_out + VF - v_in) / (v_in - VSAT)
    ton = (ton_toff * t0) / (1 + ton_toff)
    toff = t0 - ton
    cap_t = 40e-6 * ton
    ipk = 2 * i_out * (ton_toff + 1)
    r_sc = 0.3 / ipk

    l_min = ton * ((v_in - VSAT) / ipk)

    cap_o = 9 * i_out * ton / v_ripple    
    return {
        "duty": ton / t0,
        "ton": ton,
        "toff": toff,
        "cap_t": cap_t,
        "ipk": ipk,
        "r_sc": r_sc,
        "l_min": l_min,
        "cap_o": cap_o
    }

mc34063_calculations(v_in, v_out, i_out, f0, v_ripple)

{'duty': 0.8008898776418243,
 'ton': 3.203559510567297e-05,
 'toff': 7.96440489432703e-06,
 'cap_t': 1.281423804226919e-09,
 'ipk': 5.022346368715083,
 'r_sc': 0.05973303670745273,
 'l_min': 2.2835428315480933e-05,
 'cap_o': 0.0014416017797552838}

In [ ]:
r2_calc = lambda r1: (v_out / 1.25 - 1) * r1

r2 = r1.apply(r2_calc)
pd.DataFrame({
    "R1 (Ohm)": r1,
    "R2 (Ohm)": r2
}).set_index("R1 (Ohm)").round(2)

,R2 (Ohm)
R1 (Ohm),
1000,14200.0
1500,21300.0
2200,31240.0
3300,46860.0
4700,66740.0
5100,72420.0
6800,96560.0


In [ ]:
r1 = 3300
r2 = 47000
v_out_calc = 1.25 * (1 + r2 / r1)
v_out_calc

19.335106382978722

## Calculo para Electronica de potencia
$Vref = Vo * \frac{r2}{r2+r1}$ => $\frac{Vo}{Vref} = 1 + \frac{r1}{r2}$


$r_1 = (\frac{Vo}{Vref} - 1) * r_2$

In [7]:

Vref_uc3843 = 2.5
df = pd.DataFrame({'R2': r_norm, 'R1': r1})
df["R1"] = (15 / Vref_uc3843 - 1) * df["R2"]
print("Valores de R1 para UC3843:")
df

Valores de R1 para UC3843:


,R2,R1
0,1000,5000.0
1,1200,6000.0
2,1500,7500.0
3,2200,11000.0
4,3300,16500.0
5,4700,23500.0
6,5100,25500.0
7,6800,34000.0


## Gain Op Amp Gate

In [4]:
datos = [
    (0.798, 3.9),
    (0.845, 4.13),
    (1.348, 6.6),
    (2, 9.84),
    (3.13, 15.4),
    (3.19, 15.6)
]
df_fet = pd.DataFrame(datos, columns=["Vpwm", "Vds"])
df_fet["Gain"] = df_fet["Vds"] / df_fet["Vpwm"]
df_fet

,Vpwm,Vds,Gain
0,0.798,3.90,4.887218
1,0.845,4.13,4.887574
2,1.348,6.60,4.896142
3,2.000,9.84,4.920000
4,3.130,15.40,4.920128
5,3.190,15.60,4.890282
